##### MemorySaver(), checkpointers, and thread_id -> to configure the memory in langGraph 

In [ ]:
from langgraph.graph import StateGraph,START,MessagesState
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from langgraph.checkpoint.memory import (MemorySaver)

In [3]:
llm = ChatGroq(model = 'llama-3.3-70b-versatile')

In [10]:
def chatbot(state: MessagesState):

    response = llm.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [11]:
builder = StateGraph(MessagesState)

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_edge(
    START,
    "chatbot"
)

In [12]:
memory = MemorySaver()
# MemorySaver stores graph state in memory.

In [13]:
"""
Without MemorySaver:
    invoke()
    ↓
    Run
    ↓
    Forget Everything

With MemorySaver:
    invoke()
    ↓
    Load Previous State
    ↓
    Run
    ↓
    Save Updated State
"""


'\nWithout MemorySaver:\n    invoke()\n    ↓\n    Run\n    ↓\n    Forget Everything\n\nWith MemorySaver:\n    invoke()\n    ↓\n    Load Previous State\n    ↓\n    Run\n    ↓\n    Save Updated State\n'

In [14]:
graph = builder.compile(checkpointer=memory)

### checkpoiter -> Enables Memory -> VV Imp

In [15]:
### Similar to session_id in RAG, here we have thread_id

In [16]:
config = {
    "configurable": {
        "thread_id": "chat_1"
    }
}

In [17]:
result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Vikas"
            )
        ]
    },
    config=config
)

In [18]:
print(result["messages"][-1].content)

Nice to meet you, Vikas. Is there something I can help you with or would you like to chat?


In [19]:
result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is my name?"
            )
        ]
    },
    config=config
)

print(result["messages"][-1].content)

Your name is Vikas.


In [20]:
state = graph.get_state(config)

for message in state.values["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 40)

HumanMessage
My name is Vikas
----------------------------------------
AIMessage
Nice to meet you, Vikas. Is there something I can help you with or would you like to chat?
----------------------------------------
HumanMessage
What is my name?
----------------------------------------
AIMessage
Your name is Vikas.
----------------------------------------


In [21]:
## different thread_d

config_2 = {
    "configurable": {
        "thread_id": "chat_2"
    }
}

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Who am I?"
            )
        ]
    },
    config=config_2
)

print(result["messages"][-1].content)

Unfortunately, I don't have any information about you, so I'll have to ask some questions to try to figure out who you are. Here's my first question:

Are you a real person, a fictional character, or something else entirely?
